## 1) 安装与导入依赖

In [34]:
# 如环境已安装可注释掉本单元
# %pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# %pip install -q opencv-python numpy trimesh ipywidgets

import os
import sys
import glob
import gc
import time
import tempfile
from pathlib import Path

import cv2
import torch
import numpy as np

# 关键：保证可导入 vggt 代码
PROJECT_ROOT = Path('/home/viplab/liukejia/project/vggt').resolve()
os.chdir(PROJECT_ROOT)
sys.path.append(str(PROJECT_ROOT))
sys.path.append(str(PROJECT_ROOT / 'vggt'))

from visual_util import predictions_to_glb
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
from vggt.utils.geometry import unproject_depth_map_to_point_map

print('PROJECT_ROOT =', PROJECT_ROOT)

PROJECT_ROOT = /home/viplab/liukejia/project/vggt


## 2) 参数配置（输入固定为 4 张图）

In [55]:
# ===== 必填参数 =====
num=8
input_image_paths = [
    f'/data/test_grid4_from_test_grid4/render_3d/{num}_0_0_0_render_3d.jpg',
    f'/data/test_grid4_from_test_grid4/render_3d/{num}_0_0_90_render_3d.jpg',
    f'/data/test_grid4_from_test_grid4/render_3d/{num}_0_0_180_render_3d.jpg',
    f'/data/test_grid4_from_test_grid4/render_3d/{num}_0_0_270_render_3d.jpg',
]

# 输出目录与结果名
output_path = '/home/viplab/liukejia/project/vggt/outputs_glb_grid4'
output_stem = f'{num}'

# ===== 处理参数 =====
force_resize = True
conf_thres = 50.0
prediction_mode = 'Depthmap and Camera Branch'
save_npz = False

valid_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

print('input_image_paths =', input_image_paths)
print('output_path =', output_path)

input_image_paths = ['/data/test_grid4_from_test_grid4/render_3d/8_0_0_0_render_3d.jpg', '/data/test_grid4_from_test_grid4/render_3d/8_0_0_90_render_3d.jpg', '/data/test_grid4_from_test_grid4/render_3d/8_0_0_180_render_3d.jpg', '/data/test_grid4_from_test_grid4/render_3d/8_0_0_270_render_3d.jpg']
output_path = /home/viplab/liukejia/project/vggt/outputs_glb_grid4


## 3) 加载 VGGT 模型

In [36]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', device)

print('Initializing VGGT...')
model = VGGT()
_URL = 'https://huggingface.co/facebook/VGGT-1B/resolve/main/model.pt'
state_dict = torch.hub.load_state_dict_from_url(_URL, map_location='cpu')
model.load_state_dict(state_dict)
model.eval().to(device)

if torch.cuda.is_available():
    major_cc = torch.cuda.get_device_capability()[0]
    amp_dtype = torch.bfloat16 if major_cc >= 8 else torch.float16
else:
    amp_dtype = None

print('amp_dtype =', amp_dtype)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

device = cuda
Initializing VGGT...
amp_dtype = torch.bfloat16


## 4) 数据准备与推理函数

In [37]:
def collect_four_input_images(input_image_paths, valid_exts=None):
    if valid_exts is None:
        valid_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

    images = [Path(p) for p in input_image_paths]
    if len(images) != 4:
        raise ValueError(f'需要恰好 4 张输入图，当前为 {len(images)} 张')

    for p in images:
        if not p.exists():
            raise FileNotFoundError(f'image not found: {p}')
        if p.suffix.lower() not in valid_exts:
            raise ValueError(f'Unsupported file extension: {p.suffix} ({p})')

    return images


def read_single_image_512(image_path: str, force_resize: bool = True):
    image_path = Path(image_path)
    bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise ValueError(f'Failed to read image: {image_path}')

    h, w = bgr.shape[:2]
    if (h, w) != (512, 512):
        if force_resize:
            bgr = cv2.resize(bgr, (512, 512), interpolation=cv2.INTER_AREA)
            print(f'[Info] resized {image_path.name}: {(w, h)} -> (512, 512)')
        else:
            raise ValueError(f'{image_path.name}: expect 512x512, got {(w, h)}')

    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def save_four_images_to_target(image_paths, target_dir: str, force_resize: bool = True):
    target_dir = Path(target_dir)
    images_dir = target_dir / 'images'
    images_dir.mkdir(parents=True, exist_ok=True)

    saved_paths = []
    for i, img_path in enumerate(image_paths):
        rgb = read_single_image_512(str(img_path), force_resize=force_resize)
        out_path = images_dir / f'{i:06d}.png'
        cv2.imwrite(str(out_path), cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
        saved_paths.append(str(out_path))

    return saved_paths


def run_model(target_dir: str, model) -> dict:
    target_dir = Path(target_dir)
    image_names = sorted(glob.glob(str(target_dir / 'images' / '*')))
    if len(image_names) == 0:
        raise ValueError(f'No images found in {target_dir / "images"}')

    images = load_and_preprocess_images(image_names).to(device)

    with torch.no_grad():
        if torch.cuda.is_available():
            with torch.cuda.amp.autocast(dtype=amp_dtype):
                predictions = model(images)
        else:
            predictions = model(images)

    extrinsic, intrinsic = pose_encoding_to_extri_intri(predictions['pose_enc'], images.shape[-2:])
    predictions['extrinsic'] = extrinsic
    predictions['intrinsic'] = intrinsic

    for key in list(predictions.keys()):
        if isinstance(predictions[key], torch.Tensor):
            predictions[key] = predictions[key].detach().cpu().numpy().squeeze(0)

    predictions['pose_enc_list'] = None

    depth_map = predictions['depth']
    world_points = unproject_depth_map_to_point_map(
        depth_map,
        predictions['extrinsic'],
        predictions['intrinsic']
    )
    predictions['world_points_from_depth'] = world_points

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return predictions


def export_npz_and_glb(
    sample_target_dir: str,
    predictions: dict,
    output_dir: str,
    output_stem: str,
    conf_thres: float = 50.0,
    prediction_mode: str = 'Depthmap and Camera Branch',
    save_npz: bool = True,
):
    sample_target_dir = Path(sample_target_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    glb_path = output_dir / f'{output_stem}.glb'
    npz_path = output_dir / f'{output_stem}.npz'

    if save_npz:
        np.savez(npz_path, **predictions)

    glbscene = predictions_to_glb(
        predictions,
        conf_thres=conf_thres,
        filter_by_frames='All',
        show_cam=True,
        mask_sky=False,
        mask_black_bg=False,
        mask_white_bg=False,
        target_dir=str(sample_target_dir),
        prediction_mode=prediction_mode,
    )
    glbscene.export(file_obj=str(glb_path))

    return (str(npz_path) if save_npz else None), str(glb_path)


def reconstruct_from_four_images(
    input_image_paths,
    output_path: str,
    output_stem: str = 'sample_4views',
    conf_thres: float = 50.0,
    prediction_mode: str = 'Depthmap and Camera Branch',
    force_resize: bool = True,
    save_npz: bool = True,
    valid_exts=None,
):
    t0 = time.time()
    output_dir = Path(output_path)
    output_dir.mkdir(parents=True, exist_ok=True)

    image_paths = collect_four_input_images(input_image_paths, valid_exts=valid_exts)

    sample_target_dir = Path(tempfile.mkdtemp(prefix=f'vggt_{output_stem}_'))
    _ = save_four_images_to_target(image_paths, sample_target_dir, force_resize=force_resize)

    predictions = run_model(sample_target_dir, model)
    npz_path, glb_path = export_npz_and_glb(
        sample_target_dir=sample_target_dir,
        predictions=predictions,
        output_dir=output_dir,
        output_stem=output_stem,
        conf_thres=conf_thres,
        prediction_mode=prediction_mode,
        save_npz=save_npz,
    )

    elapsed = time.time() - t0
    print(f'[Done] elapsed: {elapsed:.2f}s')

    return {
        'input_images': [str(p) for p in image_paths],
        'target_dir': str(sample_target_dir),
        'glb_path': glb_path,
        'npz_path': npz_path,
    }

## 5) 一键运行

In [56]:
result = reconstruct_from_four_images(
    input_image_paths=input_image_paths,
    output_path=output_path,
    output_stem=output_stem,
    conf_thres=conf_thres,
    prediction_mode=prediction_mode,
    force_resize=force_resize,
    save_npz=save_npz,
    valid_exts=valid_exts,
)

print('GLB:', result['glb_path'])
print('NPZ:', result['npz_path'])
result

Building GLB scene
Using Depthmap and Camera Branch
GLB Scene built
[Done] elapsed: 1.05s
GLB: /home/viplab/liukejia/project/vggt/outputs_glb_grid4/8.glb
NPZ: None


{'input_images': ['/data/test_grid4_from_test_grid4/render_3d/8_0_0_0_render_3d.jpg',
  '/data/test_grid4_from_test_grid4/render_3d/8_0_0_90_render_3d.jpg',
  '/data/test_grid4_from_test_grid4/render_3d/8_0_0_180_render_3d.jpg',
  '/data/test_grid4_from_test_grid4/render_3d/8_0_0_270_render_3d.jpg'],
 'target_dir': '/tmp/vggt_8_jnzq2_zo',
 'glb_path': '/home/viplab/liukejia/project/vggt/outputs_glb_grid4/8.glb',
 'npz_path': None}